# Autonomous Lead Generation Agent (LangGraph + Groq)

Give this agent a plain-English request such as:

> "find all scientists working on agentic AI these days"

and it will **search the web, visit real pages, pull out names/emails/organizations, and save everything to `leads.csv`** — on its own, looping between tools until it has a good list.

### What's different from a "fake" agent
This uses LangGraph's `tools_condition` for real conditional routing: the LLM itself decides, at every step, whether to call a tool again or stop. There is no hardcoded "search -> extract -> save" sequence — the model can search multiple times, follow links, retry with a better query, etc.

**You need one free API key:** [console.groq.com/keys](https://console.groq.com/keys) (Groq is free and gives fast, reliable tool-calling models — a local CPU GGUF model is not reliable enough for multi-step tool use).


In [ ]:
!pip install -q langgraph langchain-core langchain-groq duckduckgo-search beautifulsoup4 requests pydantic

## 1. Set your Groq API key

Get a free key at https://console.groq.com/keys, then paste it below (or set it as an environment variable before starting Jupyter).

In [ ]:
import os
import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

print("Groq key set." if os.environ.get("GROQ_API_KEY") else "No key set!")

## 2. Imports and config

In [ ]:
import re
import csv
import time
from typing import List

import requests
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS

from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

GROQ_MODEL = "llama-3.3-70b-versatile"
OUTPUT_CSV = "leads.csv"
MAX_AGENT_STEPS = 40

EMAIL_REGEX = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+")
print("Config loaded.")

## 3. Tools

The tool **docstrings are the prompt** the model sees when deciding what to call — this is exactly what was weak in the earlier version, so each one below is written to be specific about *when* to use it and *what* it returns.

- `web_search` — finds candidate pages (faculty pages, lab team pages, papers, etc.)
- `scrape_website` — reads a specific page and auto-detects any emails on it
- `save_leads` — the agent's own structured "final answer" tool; it must call this once, with a validated list of `{name, email, organization, source_url}`, instead of us regex-parsing free text afterwards


In [ ]:
@tool
def web_search(query: str, max_results: int = 8) -> str:
    """Search the public web for pages related to a topic, field, or role.

    Use this FIRST to discover candidate pages that likely list real
    people's names and contact info: university faculty/team pages,
    research lab "People" pages, personal academic websites, conference
    speaker lists, company "About us" pages, recent papers with author
    affiliations, etc.

    Args:
        query: A focused search query. Prefer specific phrasing such as
            "<topic> research lab team" or "<topic> faculty directory"
            or "<topic> researchers 2026" rather than a raw copy of the
            user's full sentence.
        max_results: How many results to fetch (default 8, max 15).

    Returns:
        A numbered list of results, each with a title, URL, and short
        snippet. Use the URLs with the scrape_website tool.
    """
    max_results = max(1, min(max_results, 15))
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
    except Exception as e:
        return f"SEARCH_ERROR: {e}"

    if not results:
        return "NO_RESULTS: try a different / broader query."

    lines = []
    for i, r in enumerate(results, 1):
        title = r.get("title", "").strip()
        href = r.get("href", "").strip()
        body = r.get("body", "").strip()
        lines.append(f"{i}. {title}\n   URL: {href}\n   Snippet: {body}")
    return "\n\n".join(lines)

In [ ]:
@tool
def scrape_website(url: str) -> str:
    """Fetch a specific web page and read its visible text content.

    Use this on URLs returned by web_search (or on other URLs you find
    while reading pages, e.g. a "Contact" or "Team" link) to look for a
    person's full name, title/organization, and email address.

    Args:
        url: The exact page URL to fetch. Must start with http:// or https://

    Returns:
        The page's visible text (truncated to a reasonable length) plus a
        separate "EMAILS_FOUND_ON_PAGE" line listing any email addresses
        detected on the page via pattern matching. If no emails are
        detected, that line will say "none".
    """
    if not url.startswith(("http://", "https://")):
        return "SCRAPE_ERROR: url must start with http:// or https://"

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
        )
    }
    try:
        resp = requests.get(url, headers=headers, timeout=12)
        resp.raise_for_status()
    except Exception as e:
        return f"SCRAPE_ERROR: could not fetch {url} ({e})"

    soup = BeautifulSoup(resp.text, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = re.sub(r"\n{2,}", "\n", soup.get_text("\n")).strip()
    emails = sorted(set(EMAIL_REGEX.findall(resp.text)))

    text_truncated = text[:4000]
    emails_line = ", ".join(emails) if emails else "none"

    return (
        f"PAGE_TEXT (truncated):\n{text_truncated}\n\n"
        f"EMAILS_FOUND_ON_PAGE: {emails_line}"
    )

In [ ]:
class Lead(BaseModel):
    name: str = Field(description="Full name of the person.")
    email: str = Field(
        description="Email address for this person. Use 'not_found' if none was found."
    )
    organization: str = Field(
        description="University, company, or lab the person is affiliated with. "
        "Use 'unknown' if not stated."
    )
    source_url: str = Field(description="The URL where this lead's info was found.")


class SaveLeadsInput(BaseModel):
    leads: List[Lead] = Field(
        description="The complete, deduplicated list of leads gathered so far."
    )


@tool(args_schema=SaveLeadsInput)
def save_leads(leads: List[Lead]) -> str:
    """FINAL STEP ONLY. Save the complete list of collected leads to a CSV file.

    Call this exactly ONCE, only after you are done searching and scraping,
    passing every real lead you found across all pages (deduplicated by
    name+email). Do not call this early with a partial list, and do not
    call it more than once.

    Only include entries with a real, specific person's full name (not a
    lab name, not a generic department). If you found a name but no email,
    still include the row with email set to "not_found".
    """
    rows = [l.model_dump() if hasattr(l, "model_dump") else dict(l) for l in leads]

    seen = set()
    unique_rows = []
    for r in rows:
        key = (r["name"].strip().lower(), r["email"].strip().lower())
        if key in seen:
            continue
        seen.add(key)
        unique_rows.append(r)

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f, fieldnames=["name", "email", "organization", "source_url"]
        )
        writer.writeheader()
        writer.writerows(unique_rows)

    return f"Saved {len(unique_rows)} unique leads to {OUTPUT_CSV}."


TOOLS = [web_search, scrape_website, save_leads]
print("Tools ready:", [t.name for t in TOOLS])

## 4. System prompt + LLM

This is the agent's instructions for *how* to use the tools — the other big weakness in the earlier version, which only told the local model to output a bullet list rather than actually plan a research loop.

In [ ]:
SYSTEM_PROMPT = """You are an autonomous Lead Generation Research Agent.

The user will describe a topic, field, or type of role (for example:
"scientists working on agentic AI" or "startups doing solar panel recycling").
Your job is to find REAL people currently active in that space and collect
their Name, Email, Organization, and the Source URL where you found them.

You have three tools: web_search, scrape_website, and save_leads.

Follow this loop:
1. Call web_search with a focused query to find pages likely to list real
   people (lab "team" pages, university faculty directories, recent papers,
   conference speaker pages, company "about us" pages).
2. For each promising URL in the results, call scrape_website to read it and
   look for full names, organizations, and email addresses.
3. If a page names people but has no email, try another web_search such as
   "<person name> <organization> email" or "<person name> contact page",
   then scrape_website that result too.
4. Keep searching and scraping with varied queries until you have gathered
   as many distinct, real people as you reasonably can (aim for at least
   8-15 if the topic has enough coverage; fewer is fine if the topic is
   niche - never invent people to hit a number).
5. When you are done gathering, call save_leads exactly once with the full,
   deduplicated list.

Hard rules:
- NEVER invent, guess, or hallucinate a name or email. Only use what tools
  actually returned. If you cannot find an email for someone, set it to
  "not_found" but still include their name and organization.
- Do not call save_leads more than once, and do not call it until you have
  finished researching.
- Prefer primary sources (university/lab/company pages) over aggregator or
  social-media snippets.
- Keep working autonomously - do not ask the user clarifying questions;
  make reasonable interpretations of the request yourself.
"""

llm = ChatGroq(model=GROQ_MODEL, temperature=0.1)
llm_with_tools = llm.bind_tools(TOOLS)
print("LLM ready:", GROQ_MODEL)

## 5. Build the graph

This is the real agentic part: `tools_condition` looks at the model's last message and routes to `tools` if it made a tool call, or to `END` if it's done. That's the actual loop shown in the architecture diagram earlier in this notebook — not a fixed 3-node pipeline.

In [ ]:
def agent_node(state: MessagesState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(TOOLS))

graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")

compiled_agent = graph.compile()
print("Lead Generation Agent compiled and ready.")

## 6. Run it

Edit the request below to whatever you want leads for.

In [ ]:
user_request = "please find all scientists that work in agentic AI these days"

messages = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=user_request)]

print(f"Running agent for: {user_request!r}\n")

for step in compiled_agent.stream(
    {"messages": messages},
    {"recursion_limit": MAX_AGENT_STEPS},
    stream_mode="values",
):
    last = step["messages"][-1]
    role = last.__class__.__name__
    tool_calls = getattr(last, "tool_calls", None)
    if tool_calls:
        for tc in tool_calls:
            print(f"[{role}] -> calling tool: {tc['name']}({tc['args']})")
    else:
        content = str(getattr(last, "content", ""))[:300]
        if content:
            print(f"[{role}] {content}")

## 7. Check the results

In [ ]:
import pandas as pd
import os

if os.path.exists(OUTPUT_CSV):
    df = pd.read_csv(OUTPUT_CSV)
    print(f"{len(df)} leads saved to {OUTPUT_CSV}")
    display(df)
else:
    print("No leads.csv found yet - the agent may not have finished. Check the log above.")